# Images and Vision
The basic idea is to pass an image as part of the prompt. This is done by specifying the content type as `image_url` of the `message` in the input JSON. The image can be passed as a URL or a base64 encoded file (the content type in both cases remains the same). I can provide multiple images in the prompt and then ask the LLM questions about the image.

If I specify "low" detail, the image is converted into 512x512 and then its 85 token embedding (is this the same as an 85D embedding?) is fed to the model. If I specify "high" detail, then in addition to embedding of the entire 512x512 image, the API will cut the image up in 512x512 tiles with each tile having a 170 token embedding that are all fed into the model. Read more about the exact methodology to calculate cost [here](https://platform.openai.com/docs/guides/images?api-mode=chat&format=base64-encoded#calculating-costs).

### Limitations
  * Medical images don't work.
  * Trying to identify non-english text inside the image won't work.
  * If I want to read the English text in the image, I should enlarge the image.
  * Rotated (upside down) images might confuse the model.
  * Graphs don't seem to work too well.
  * Spatial reasoning, e.g., identifying chess positions, does not seem to work too well.
  * Panoramic or fish eye images don't work.
  * Counting inside the image does not work too well.
  * CAPTCHAS are blocked.

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image
from pathlib import Path
import base64
from utils import LLM

In [2]:
DATAROOT = Path.home() / "mldata"

In [3]:
load_dotenv()

True

In [4]:
client = OpenAI()

In [5]:
def encode_image(image_file: Path) -> str:
    with open(image_file, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [6]:
b64img = encode_image(DATAROOT / "IMG_0682.png")
completion = client.chat.completions.create(
    model=LLM.PRE_FAST_MINI,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "This image has my breakfast. What did I have for breakfast?",
                },
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{b64img}"},
                },
            ],
        }
    ],
)
print(completion.choices[0].message.content)

For breakfast, it looks like you had two slices of toasted bread and some kind of egg dish, possibly an omelet or scrambled eggs. You also have a cup of coffee. Enjoy your meal!


In [8]:
b64img1 = encode_image(DATAROOT / "IMG_0570.png")
b64img2 = encode_image(DATAROOT / "IMG_0747.png")

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "A total of how many people are in these images?",
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{b64img1}",
                    "detail": "low",
                },
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{b64img2}",
                    "detail": "low",
                },
            },
        ],
    }
]

In [ ]:
completion = client.chat.completions.create(
    model=LLM.PRE_NANO,
    messages=messages,  # type: ignore
)
print(completion.choices[0].message.content)

There are a total of four people in these images.


In [9]:
completion = client.chat.completions.create(
    model=LLM.PRE,
    messages=messages,  # type: ignore
)
print(completion.choices[0].message.content)

There are a total of **two people** in these images. Both images feature the same two individuals.
